# Measuring Earth's rotation with a pendulum
A pendulum swings in seconds but its direction turns over hours. Predict the sign of the change before running the cells. The origin is the suspension point projected onto the floor. Angles are positive counterclockwise as viewed from above.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

latitude_deg = 48.0    # try 0, -48, and 90
length_m = 10.0
amplitude_m = 0.15
hours_between = 3.0
assert -90 <= latitude_deg <= 90 and length_m > 0 and 0 < amplitude_m < 0.1*length_m
assert 0 < hours_between <= 12

## Separate fast and slow motion
For east and north coordinates combine the horizontal equations into a complex displacement. An eastward release without initial velocity requires a small imaginary correction even when the observer sees an almost straight trace.

In [ ]:
earth_rate = 2*np.pi/86164.1
f = earth_rate*np.sin(np.deg2rad(latitude_deg))
omega = np.sqrt(9.81/length_m)
nu = np.hypot(omega, f)
def displacement(time):
    time = np.asarray(time)
    return amplitude_m*np.exp(-1j*f*time)*(np.cos(nu*time) + 1j*(f/nu)*np.sin(nu*time))

period_s = 2*np.pi/nu
days = 2*np.pi/(abs(f)*86400) if abs(f)>0 else np.inf
print(f"Swing period {period_s:.4f} s; apparent drift {-np.rad2deg(f)*3600:.3f} deg/h")
print("Directed-axis rotation period (solar days):", days)

## Compare floor traces with the latitude law
Each coloured curve spans one full swing at the displayed clock time. The adjacent graph uses a directed angle. A straight line on its own does not distinguish angles separated by 180 degrees.

In [ ]:
fig, axes = plt.subplots(1,2,figsize=(9,3.9),layout="constrained")
for hour, colour in zip([0,hours_between,2*hours_between],["#175868","#a23d21","#375b9a"]):
    u=displacement(hour*3600+np.linspace(0,period_s,220))
    axes[0].plot(u.real*100,u.imag*100,lw=2.4,color=colour,label=f"{hour:g} h")
axes[0].set(xlabel="East (cm)",ylabel="North (cm)",aspect="equal",title="One swing at three times",
            xlim=(-110*amplitude_m,110*amplitude_m),ylim=(-110*amplitude_m,110*amplitude_m))
axes[0].legend(fontsize=9)
angles=np.linspace(-90,90,181); rates=-np.rad2deg(earth_rate*np.sin(np.deg2rad(angles)))*3600
axes[1].plot(angles,rates,color="#175868");axes[1].plot(latitude_deg,-np.rad2deg(f)*3600,"o",color="#a23d21")
axes[1].axhline(0,lw=.6,color="grey")
axes[1].set(xlabel="Latitude (degrees north)",ylabel="Azimuth drift (degrees per hour)",title="Clockwise drift is negative")
plt.show()

## Verify the initial condition and the equatorial limit
Differentiate the sampled position symmetrically. Repeat with zero vertical rotation rate. Explain why changing the length changes the fast period but leaves the azimuthal drift rate unchanged.

In [ ]:
assert abs(displacement(0)-amplitude_m)<1e-12
assert abs((displacement(1e-5)-displacement(-1e-5))/2e-5)<1e-8
if latitude_deg == 0: assert abs(displacement(.31).imag)<1e-12

## Optional OneSimulation check
In a OneSimulation installation, integrate both coupled real equations without using the complex solution. Compare the result after three swings. In the web notebook the analytic study above is fully usable with NumPy alone.

In [ ]:
try:
    from onesim.numerics.time import ClassicalRungeKutta4TimeIntegrator
except ImportError:
    print("The optional integrator check runs in a local OneSimulation installation.")
else:
    def acceleration(s,t):
        x,y,vx,vy=s
        return (vx,vy,-omega**2*x+2*f*vy,-omega**2*y-2*f*vx)
    state=(amplitude_m,0.0,0.0,0.0); dt=period_s/180
    method=ClassicalRungeKutta4TimeIntegrator()
    for j in range(540): state=method.advance(state,acceleration,dt,time=j*dt)
    error=abs(complex(*state[:2])-displacement(540*dt))/amplitude_m
    print("Relative final displacement disagreement:",error)
    assert error < 1e-5